
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Lecture - Agent Deployment on Databricks
#### Duration: 20 minutes

## Overview
This lecture covers the architecture and tooling required to move a generative AI agent from a notebook prototype into a production Databricks App. The focus is on the agent lifecycle, Databricks Apps as the deployment target, and Declarative Automation Bundles (DABs) as the packaging and deployment mechanism.

## Learning Objectives
By the end of this lecture you will be able to:
- Outline the four stages of the agent lifecycle on Databricks
- Differentiate Databricks Apps deployment from model serving endpoint deployment
- Explain how Declarative Automation Bundles structure and deploy an agent application
- Describe how resource injection decouples agent code from infrastructure configuration
- Identify the role of Unity Gateway as the governance layer for deployed agents

## A. The Agent Lifecycle

### A1. From Prototype to Production

Building an agent in a notebook is only the beginning. A production-grade agent must be **deployed** to infrastructure that can serve requests, **observed** so you can understand its behavior, **evaluated** to measure quality, and **monitored** continuously to catch regressions before users notice them.

On Databricks, this lifecycle maps to four stages:

<div style="background: #fff; max-width: 1600px; margin: 0 auto; padding: 12px 0 24px;">
<svg viewBox="0 0 1600 187.55" width="100%" role="img" aria-label="The agent lifecycle on Databricks: deploy with Databricks Apps, observe with MLflow Tracing, evaluate with scorers and judges, and monitor with backfill and SQL queries over traces in Unity Catalog." style="display: block; height: auto; font-family: -apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Arial,sans-serif;">
<defs>
<filter id="lc-sh" x="-10%" y="-10%" width="120%" height="130%"><feDropShadow dx="0" dy="3" stdDeviation="6" flood-color="#1B3139" flood-opacity="0.08"/></filter>
</defs>
<rect x="40" y="20" width="340" height="148" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#lc-sh)"/>
<rect x="62" y="34" width="296" height="5" rx="2.5" fill="#FF5F46"/>
<rect x="62" y="53" width="79" height="26" rx="13.0" fill="#FF5F46"/><text x="101" y="71" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">DEPLOY</text>
<text x="62" y="112" font-size="21" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">Databricks Apps</text>
<text x="62" y="146" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">Ship the agent with a bundle</text>
<rect x="433" y="20" width="340" height="148" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#lc-sh)"/>
<rect x="455" y="34" width="296" height="5" rx="2.5" fill="#2272B4"/>
<rect x="455" y="53" width="88" height="26" rx="13.0" fill="#2272B4"/><text x="499" y="71" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">OBSERVE</text>
<text x="455" y="112" font-size="21" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">MLflow Tracing</text>
<text x="455" y="146" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">Trace every request</text>
<rect x="826" y="20" width="340" height="148" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#lc-sh)"/>
<rect x="848" y="34" width="296" height="5" rx="2.5" fill="#00A972"/>
<rect x="848" y="53" width="97" height="26" rx="13.0" fill="#00A972"/><text x="896" y="71" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">EVALUATE</text>
<text x="848" y="112" font-size="21" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">Scorers and judges</text>
<text x="848" y="146" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">Score traces as they arrive</text>
<rect x="1219" y="20" width="340" height="148" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#lc-sh)"/>
<rect x="1241" y="34" width="296" height="5" rx="2.5" fill="#1B5162"/>
<rect x="1241" y="53" width="88" height="26" rx="13.0" fill="#1B5162"/><text x="1285" y="71" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">MONITOR</text>
<text x="1241" y="112" font-size="21" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">Backfill and query</text>
<text x="1241" y="146" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">Rescore and query history</text>
</svg>
</div>

- **Deployment**: Package the agent and deploy it as a Databricks App using Declarative Automation Bundles (DABs). The app runs as a managed web application with its own compute, service principal, and URL.
- **Observability**: Integrate MLflow Tracing so every request produces a structured trace capturing inputs, outputs, latency, and execution flow.
- **Evaluation**: Attach scorers to the MLflow Experiment that automatically assess trace quality using LLM judges, guidelines, or custom Python functions.
- **Monitoring**: Manage scorer lifecycles, backfill historical traces with new scorers, and query traces stored in Unity Catalog with SQL for long-term analysis.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">These stages are not strictly sequential. You can add tracing before deployment, attach scorers to an agent that is already live, or backfill evaluation data onto traces logged weeks ago. The lifecycle is designed to be incremental.</p>
</div>
</div>
</div>

Databricks Apps key concepts (<a href="https://docs.databricks.com/aws/en/dev-tools/databricks-apps/key-concepts" target="_blank" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/dev-tools/databricks-apps/key-concepts" target="_blank" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/dev-tools/databricks-apps/key-concepts" target="_blank" style="color: #1976d2; text-decoration: underline;">GCP</a>)

## B. Deployment Architecture

### B1. Databricks Apps

**Databricks Apps** provide a managed hosting environment for web applications running on the Databricks platform. When you deploy an agent as a Databricks App, the platform handles the infrastructure concerns so you can focus on the application logic.

Key characteristics of a Databricks App:
- **Managed compute**: The platform provisions and manages the compute resources. You do not configure clusters, VMs, or containers.
- **Service principal**: Each app runs under its own service principal identity, which determines what Databricks resources the app can access (model services, UC objects, experiments, etc).
- **Unique URL**: Every deployed app gets a workspace-scoped URL that users or downstream services can call directly.
- **Workspace integration**: Apps are first-class workspace objects visible on the **Apps** page in the workspace sidebar, where you can monitor status, view logs, and trigger redeployments.

<div style="background: #fff; max-width: 1600px; margin: 0 auto; padding: 12px 0 24px;">
<svg viewBox="0 0 1600 558" width="100%" role="img" aria-label="Anatomy of a Databricks App. The app my-agent-app has source code, managed compute, a URL, resources declared in databricks.yml, and its own service principal. A dotted line links its resources to the bundle resources granted to the service principal: EXECUTE on the agent-model service, CAN_EDIT on the my-agent-experiment MLflow experiment, SELECT on the listings table, and CAN_USE on the my_warehouse SQL warehouse." style="display: block; height: auto; font-family: -apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Arial,sans-serif;">
<defs>
<marker id="aa-ar" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="8" markerHeight="8" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#618794"/></marker>
<filter id="aa-sh" x="-10%" y="-10%" width="120%" height="130%"><feDropShadow dx="0" dy="3" stdDeviation="6" flood-color="#1B3139" flood-opacity="0.08"/></filter>
</defs>
<rect x="40" y="20" width="700" height="518" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#aa-sh)"/>
<rect x="62" y="34" width="656" height="5" rx="2.5" fill="#FF5F46"/>
<rect x="62" y="53" width="151" height="26" rx="13.0" fill="#FF5F46"/><text x="138" y="71" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">DATABRICKS APP</text>
<text x="62" y="112" font-size="21" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">my-agent-app</text>
<rect x="860" y="96" width="700" height="442" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#aa-sh)"/>
<rect x="882" y="110" width="656" height="5" rx="2.5" fill="#2272B4"/>
<rect x="882" y="129" width="170" height="26" rx="13.0" fill="#2272B4"/><text x="967" y="147" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">BUNDLE RESOURCES</text>
<text x="882" y="188" font-size="21" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">Granted to the service principal</text>
<rect x="62" y="148" width="656" height="64" rx="10" fill="#F9F7F4" stroke="#EEEDE9"/>
<text x="86" y="188" font-size="19" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">Source code</text>
<text x="340" y="188" font-size="17" font-weight="500" fill="#5A6F77" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">agent.py · server.py</text>
<rect x="62" y="224" width="656" height="64" rx="10" fill="#F9F7F4" stroke="#EEEDE9"/>
<text x="86" y="264" font-size="19" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">Compute</text>
<text x="340" y="264" font-size="17" font-weight="500" fill="#5A6F77" text-anchor="start" xml:space="preserve">Managed by Databricks</text>
<rect x="62" y="300" width="656" height="64" rx="10" fill="#F9F7F4" stroke="#EEEDE9"/>
<text x="86" y="340" font-size="19" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">URL</text>
<text x="340" y="340" font-size="17" font-weight="500" fill="#5A6F77" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">&amp;lt;app&amp;gt;.databricksapps.com</text>
<rect x="62" y="376" width="656" height="64" rx="10" fill="#F9F7F4" stroke="#EEEDE9"/>
<text x="86" y="416" font-size="19" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">Resources</text>
<text x="340" y="416" font-size="17" font-weight="500" fill="#5A6F77" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">databricks.yml</text>
<rect x="62" y="452" width="656" height="64" rx="10" fill="#F9F7F4" stroke="#EEEDE9"/>
<text x="86" y="492" font-size="19" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">Service principal</text>
<text x="340" y="492" font-size="17" font-weight="500" fill="#5A6F77" text-anchor="start" xml:space="preserve">The app’s identity</text>
<rect x="882" y="224" width="656" height="64" rx="10" fill="#F9F7F4" stroke="#EEEDE9"/>
<text x="906" y="264" font-size="19" font-weight="700" fill="#0b2026" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">agent-model</text>
<text x="1220" y="264" font-size="17" font-weight="500" fill="#5A6F77" text-anchor="start" xml:space="preserve">Model service</text>
<rect x="1418" y="243" width="110" height="26" rx="13.0" fill="#2272B4"/><text x="1473" y="261" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">EXECUTE</text>
<rect x="882" y="300" width="656" height="64" rx="10" fill="#F9F7F4" stroke="#EEEDE9"/>
<text x="906" y="340" font-size="19" font-weight="700" fill="#0b2026" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">my-agent-experiment</text>
<text x="1220" y="340" font-size="17" font-weight="500" fill="#5A6F77" text-anchor="start" xml:space="preserve">MLflow experiment</text>
<rect x="1418" y="319" width="110" height="26" rx="13.0" fill="#00A972"/><text x="1473" y="337" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">CAN_EDIT</text>
<rect x="882" y="376" width="656" height="64" rx="10" fill="#F9F7F4" stroke="#EEEDE9"/>
<text x="906" y="416" font-size="19" font-weight="700" fill="#0b2026" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">listings</text>
<text x="1220" y="416" font-size="17" font-weight="500" fill="#5A6F77" text-anchor="start" xml:space="preserve">Table</text>
<rect x="1418" y="395" width="110" height="26" rx="13.0" fill="#1B5162"/><text x="1473" y="413" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">SELECT</text>
<rect x="882" y="452" width="656" height="64" rx="10" fill="#F9F7F4" stroke="#EEEDE9"/>
<text x="906" y="492" font-size="19" font-weight="700" fill="#0b2026" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">my_warehouse</text>
<text x="1220" y="492" font-size="17" font-weight="500" fill="#5A6F77" text-anchor="start" xml:space="preserve">SQL warehouse</text>
<rect x="1418" y="471" width="110" height="26" rx="13.0" fill="#FF5F46"/><text x="1473" y="489" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">CAN_USE</text>
<path d="M748,408.0 H852" fill="none" stroke="#2272B4" stroke-width="3.5" stroke-linecap="round" stroke-dasharray="0.1 8"/>
</svg>
</div>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">In the app-based deployment pattern, the agent code itself is the deployment artifact. The app calls a model service for LLM inference, but the orchestration logic, tool integrations, and observability wiring all live in your application source.</p>
</div>
</div>
</div>


### B2. Declarative Automation Bundles

<div style="display: flex; align-items: center; gap: 32px; margin: 16px 0 24px;">
  <div style="flex-shrink: 0; position: relative; width: 180px; height: 180px;">
    <div style="position: absolute; left: 9px; top: 9px; width: 162px; height: 162px; border-radius: 22px; background: #2272B4; display: flex; align-items: center; justify-content: center;">
      <svg width="116" height="116" viewBox="0 0 150 150" fill="none" xmlns="http://www.w3.org/2000/svg" style="display:block">
      <path d="M43.25 26.4004L17.5 41.2704V71.0004H20.5V43.0004L44.75 29.0004L43.25 26.4004Z" fill="white"/>
      <path d="M75 137.4L50.75 123.4L49.25 126L75 140.86L100.75 126L99.25 123.4L75 137.4Z" fill="white"/>
      <path d="M106.75 26.4004L105.25 29.0004L129.5 43.0004V71.0004H132.5V41.2704L106.75 26.4004Z" fill="white"/>
      <path d="M23.5 44.73V104.2L75 133.93L126.5 104.2V44.73L75 15L23.5 44.73ZM97 31.16L87.97 36.38C85.08 32.32 80.35 29.66 75 29.66C69.65 29.66 64.92 32.32 62.03 36.37L53 31.16L75 18.46L97 31.16ZM112.93 88.89C112.93 96.02 107.13 101.82 100 101.82C92.87 101.82 87.07 96.02 87.07 88.89C87.07 81.76 92.87 75.96 100 75.96C107.13 75.96 112.93 81.76 112.93 88.89ZM98.5 73.03C90.42 73.79 84.07 80.61 84.07 88.89C84.07 91.26 84.6 93.49 85.53 95.51L75 101.59L64.47 95.51C65.4 93.49 65.93 91.25 65.93 88.89C65.93 80.61 59.58 73.79 51.5 73.03V60.89L62.03 54.81C64.92 58.87 69.65 61.52 75 61.52C80.35 61.52 85.08 58.86 87.97 54.81L98.5 60.89V73.03ZM50 101.82C42.87 101.82 37.07 96.02 37.07 88.89C37.07 81.76 42.87 75.96 50 75.96C57.13 75.96 62.93 81.76 62.93 88.89C62.93 96.02 57.13 101.82 50 101.82ZM87.93 45.59C87.93 52.72 82.13 58.52 75 58.52C67.87 58.52 62.07 52.72 62.07 45.59C62.07 38.46 67.87 32.66 75 32.66C82.13 32.66 87.93 38.46 87.93 45.59ZM50 32.89L60.53 38.97C59.6 40.99 59.07 43.23 59.07 45.59C59.07 47.95 59.6 50.19 60.53 52.21L50 58.29L32 47.9L30.5 50.5L48.5 60.89V73.03C43.77 73.47 39.65 76 37.03 79.67L26.5 73.59V46.45L50 32.88V32.89ZM26.5 77.07L35.53 82.28C34.6 84.3 34.07 86.54 34.07 88.9C34.07 97.18 40.42 104 48.5 104.76V115.17L26.5 102.47V77.07ZM51.5 116.91V104.77C56.23 104.33 60.35 101.8 62.97 98.13L73.5 104.21V124.99H76.5V104.21L87.03 98.13C89.65 101.8 93.77 104.32 98.5 104.77V116.91L75 130.48L51.5 116.91ZM101.5 115.18V104.77C109.58 104.01 115.93 97.19 115.93 88.91C115.93 86.55 115.4 84.31 114.47 82.29L123.5 77.08V102.48L101.5 115.18ZM123.5 73.61L112.97 79.69C110.35 76.02 106.23 73.5 101.5 73.05V60.91L119.5 50.52L118 47.92L100 58.31L89.47 52.23C90.4 50.21 90.93 47.97 90.93 45.61C90.93 43.25 90.4 41.01 89.47 38.99L100 32.91L123.5 46.48V73.62V73.61Z" fill="white"/>
      <path d="M43.5 95.4004H56.5V82.4004H43.5V95.4004ZM46.5 85.4004H53.5V92.4004H46.5V85.4004Z" fill="white"/>
      <path d="M75 39.0996C71.42 39.0996 68.5 42.0196 68.5 45.5996C68.5 49.1796 71.42 52.0996 75 52.0996C78.58 52.0996 81.5 49.1796 81.5 45.5996C81.5 42.0196 78.58 39.0996 75 39.0996ZM75 49.0996C73.07 49.0996 71.5 47.5296 71.5 45.5996C71.5 43.6696 73.07 42.0996 75 42.0996C76.93 42.0996 78.5 43.6696 78.5 45.5996C78.5 47.5296 76.93 49.0996 75 49.0996Z" fill="white"/>
      <path d="M91.59 93.7498H108.52L100.06 79.0898L91.59 93.7498ZM96.79 90.7498L100.06 85.0898L103.33 90.7498H96.79Z" fill="white"/>
      <path d="M75 72.7402L57.25 62.4902L55.75 65.0902L73.5 75.3302V95.8302H76.5V75.3302L94.25 65.0902L92.75 62.4902L75 72.7402Z" fill="white"/>
      </svg>
    </div>
  </div>
  <div style="flex: 1;">
    <p style="margin: 0 0 14px 0; font-size: 15px; color: #333;">A <strong>Declarative Automation Bundle (DAB)</strong> is the project structure that describes your app and its resources. The project folder contains two categories of files:</p>
    <p style="margin: 0 0 6px 0; font-size: 15px; color: #333;"><strong>User-authored source files</strong> (you write these):</p>
    <ul style="margin: 0 0 14px 0; padding-left: 22px; font-size: 15px; color: #333;">
        <li style="margin: 0 0 6px 0;"><code style="font-family: monospace; font-size: 14px; background: #f1f3f7; padding: 1px 5px; border-radius: 4px;">agent.py</code>: The agent logic (using the OpenAI Agents SDK pointed at a Unity Gateway model service)</li>
        <li style="margin: 0 0 6px 0;"><code style="font-family: monospace; font-size: 14px; background: #f1f3f7; padding: 1px 5px; border-radius: 4px;">server.py</code>: A FastAPI server that wraps the agent and exposes HTTP endpoints</li>
        <li style="margin: 0 0 6px 0;"><code style="font-family: monospace; font-size: 14px; background: #f1f3f7; padding: 1px 5px; border-radius: 4px;">chat.html</code>: An optional browser chat UI that <code style="font-family: monospace; font-size: 14px; background: #f1f3f7; padding: 1px 5px; border-radius: 4px;">server.py</code> serves</li>
        <li style="margin: 0 0 6px 0;"><code style="font-family: monospace; font-size: 14px; background: #f1f3f7; padding: 1px 5px; border-radius: 4px;">pyproject.toml</code>: Python dependency specification</li>
    </ul>
    <p style="margin: 0 0 6px 0; font-size: 15px; color: #333;"><strong>Configuration files</strong> (defined by the DAB/Apps spec):</p>
    <ul style="margin: 0; padding-left: 22px; font-size: 15px; color: #333;">
        <li style="margin: 0 0 6px 0;"><code style="font-family: monospace; font-size: 14px; background: #f1f3f7; padding: 1px 5px; border-radius: 4px;">requirements.txt</code>: Installs <code style="font-family: monospace; font-size: 14px; background: #f1f3f7; padding: 1px 5px; border-radius: 4px;">uv</code>, which the startup command in <code style="font-family: monospace; font-size: 14px; background: #f1f3f7; padding: 1px 5px; border-radius: 4px;">app.yaml</code> uses to run the server</li>
        <li style="margin: 0 0 6px 0;"><code style="font-family: monospace; font-size: 14px; background: #f1f3f7; padding: 1px 5px; border-radius: 4px;">databricks.yml</code>: Bundle declaration: app name, resources (model services, MLflow experiments), and permissions</li>
        <li style="margin: 0 0 6px 0;"><code style="font-family: monospace; font-size: 14px; background: #f1f3f7; padding: 1px 5px; border-radius: 4px;">app.yaml</code>: Runtime manifest: the startup command and environment variable injection via its top-level <code style="font-family: monospace; font-size: 14px; background: #f1f3f7; padding: 1px 5px; border-radius: 4px;">env:</code> section</li>
    </ul>
  </div>
</div>

Deployment is a **two-step process**. First, run `databricks bundle deploy` from the CLI to create or update the app resource and its configuration in the workspace. Then, deploy the application source code by running `databricks apps deploy &lt;app-name&gt;` from the CLI or through the **Apps** page in the workspace sidebar. The bundle deploy step manages resource declarations; the apps deploy step pushes your source code to the running infrastructure.

<details style="margin: 14px 4px; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 0 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07); font-family: sans-serif;">
<summary style="cursor: pointer; padding: 16px 0; font-weight: 700; font-size: 18px; color: #0b2026;">Example: Two-step CLI deployment</summary>
<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
<div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
  <div style="display:inline-flex;align-items:center;gap:0.45rem;">
    <svg width="20" height="17" viewBox="0 0 20 17" fill="none" style="display:inline-block;vertical-align:middle;"><rect x="1.5" y="1" width="17" height="15" rx="2.5" stroke="#5A6F77" stroke-width="2"/><path d="M5.5 6L8.5 8.5L5.5 11" stroke="#5A6F77" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/><line x1="10.5" y1="11" x2="14.5" y2="11" stroke="#5A6F77" stroke-width="1.8" stroke-linecap="round"/></svg>
    <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">Terminal</span>
  </div>
  <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
    <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
    <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
    <span class="copy-label">Copy</span>
  </button>
</div>
<div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;"># Step 1: Deploy the bundle (creates/updates resources)
$ databricks bundle deploy
<span></span>
# Step 2: Deploy the app source code
$ databricks apps deploy my-agent-app \
    --source-code-path /Workspace/Users/&amp;lt;your-username&amp;gt;/my-agent-app</div>
</div>
</details>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">The two steps are independent. You can redeploy app source code without changing resource declarations, and you can update resource configuration without redeploying code. This separation makes iterative development faster.</p>
</div>
</div>
</div>

DABs app deployment tutorial (<a href="https://docs.databricks.com/aws/en/dev-tools/bundles/apps-tutorial" target="_blank" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/dev-tools/bundles/apps-tutorial" target="_blank" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/dev-tools/bundles/apps-tutorial" target="_blank" style="color: #1976d2; text-decoration: underline;">GCP</a>)

### B3. Comparing Deployment Patterns

There are two ways to deploy a custom agent on Databricks, and they differ in what they treat the agent as.

**Model Serving treats your agent as a model.** You package the agent code and its dependencies as an MLflow model, log and register it in Unity Catalog, and Databricks hosts it behind a managed REST endpoint. The platform owns the server; you supply a predict-style function that fits a standard interface.

**Databricks Apps treats your agent as an application.** Your agent is ordinary Python code, kept in Git. It runs as a web service on managed compute, and you control the service itself: the framework, the routes, async behavior, even a UI.

Put simply, Model Serving asks *"How do I host this artifact?"* and Apps asks *"How do I run this software?"*

<div style="background: #fff; max-width: 1600px; margin: 0 auto; padding: 12px 0 24px;">
<svg viewBox="0 0 1600 266" width="100%" role="img" aria-label="An agent as a model versus an agent as an application. On Model Serving, agent code in any framework is wrapped in a model interface, logged, registered, and versioned as an MLflow model, and hosted on a serving endpoint where Databricks owns the server. On Databricks Apps, agent code in any framework is versioned in Git and runs as a web service with its own routes, async behavior, and an optional UI, deployed as a Databricks App where Databricks hosts your server code." style="display: block; height: auto; font-family: -apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Arial,sans-serif;">
<defs>
<marker id="dp-ar" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="8" markerHeight="8" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#618794"/></marker>
<filter id="dp-sh" x="-10%" y="-10%" width="120%" height="130%"><feDropShadow dx="0" dy="3" stdDeviation="6" flood-color="#1B3139" flood-opacity="0.08"/></filter>
</defs>
<rect x="40" y="40" width="142" height="26" rx="13.0" fill="#5A6F77"/><text x="111" y="58" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">MODEL SERVING</text>
<text x="40" y="96" font-size="21" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">An agent as a model</text>
<rect x="340" y="20" width="367" height="96" rx="12" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#dp-sh)"/>
<rect x="358" y="32" width="331" height="4" rx="2" fill="#5A6F77"/>
<text x="362" y="72" font-size="21" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">Agent code</text>
<text x="362" y="100" font-size="17" font-weight="500" fill="#5A6F77" text-anchor="start" xml:space="preserve">Any framework, in a model interface</text>
<rect x="767" y="20" width="367" height="96" rx="12" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#dp-sh)"/>
<rect x="785" y="32" width="331" height="4" rx="2" fill="#5A6F77"/>
<text x="789" y="72" font-size="21" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">MLflow model</text>
<text x="789" y="100" font-size="17" font-weight="500" fill="#5A6F77" text-anchor="start" xml:space="preserve">Logged, registered, and versioned</text>
<path d="M713,68.0 H759" fill="none" stroke="#618794" stroke-width="2.4" marker-end="url(#dp-ar)"/>
<rect x="1193" y="20" width="367" height="96" rx="12" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#dp-sh)"/>
<rect x="1211" y="32" width="331" height="4" rx="2" fill="#5A6F77"/>
<text x="1215" y="72" font-size="21" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">Serving endpoint</text>
<text x="1215" y="100" font-size="17" font-weight="500" fill="#5A6F77" text-anchor="start" xml:space="preserve">Databricks owns the server</text>
<path d="M1139,68.0 H1185" fill="none" stroke="#618794" stroke-width="2.4" marker-end="url(#dp-ar)"/>
<rect x="40" y="170" width="160" height="26" rx="13.0" fill="#FF5F46"/><text x="120" y="188" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">DATABRICKS APPS</text>
<text x="40" y="226" font-size="21" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">An agent as an application</text>
<rect x="340" y="150" width="367" height="96" rx="12" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#dp-sh)"/>
<rect x="358" y="162" width="331" height="4" rx="2" fill="#FF5F46"/>
<text x="362" y="202" font-size="21" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">Agent code</text>
<text x="362" y="230" font-size="17" font-weight="500" fill="#5A6F77" text-anchor="start" xml:space="preserve">Any framework, versioned in Git</text>
<rect x="767" y="150" width="367" height="96" rx="12" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#dp-sh)"/>
<rect x="785" y="162" width="331" height="4" rx="2" fill="#FF5F46"/>
<text x="789" y="202" font-size="21" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">Web service</text>
<text x="789" y="230" font-size="17" font-weight="500" fill="#5A6F77" text-anchor="start" xml:space="preserve">Routes, async, optional UI</text>
<path d="M713,198.0 H759" fill="none" stroke="#618794" stroke-width="2.4" marker-end="url(#dp-ar)"/>
<rect x="1193" y="150" width="367" height="96" rx="12" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#dp-sh)"/>
<rect x="1211" y="162" width="331" height="4" rx="2" fill="#FF5F46"/>
<text x="1215" y="202" font-size="21" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">Databricks App</text>
<text x="1215" y="230" font-size="17" font-weight="500" fill="#5A6F77" text-anchor="start" xml:space="preserve">Databricks hosts your server code</text>
<path d="M1139,198.0 H1185" fill="none" stroke="#618794" stroke-width="2.4" marker-end="url(#dp-ar)"/>
</svg>
</div>

| Concept | Model Serving | Databricks Apps |
|---|---|---|
| **Unit of deployment** | A versioned model artifact | A codebase or application |
| **Lifecycle** | Log, register, and deploy an endpoint | Edit the code and deploy the app (Git-driven CI/CD) |
| **Versioning** | Model versions in the registry | Git commits and branches |
| **Who owns the server code** | Databricks (you fit its interface) | You (custom routes and middleware), and Databricks hosts it |
| **Iteration speed** | Slower: each change means repackaging and redeploying | Fast: change the code, redeploy in seconds, and debug locally |
| **Shape of an interaction** | Request and response | Can be long-running, async, or streaming |
| **Identity** | The endpoint owner's credentials (automatic passthrough), or on behalf of the user | The app's service principal, the user's identity, or both |
| **Front end** | Separate, such as the Review App | Can ship a chat UI alongside the agent |
| **Mental model** | An inference endpoint | A web service |

**Why the shift toward Apps.** Agents look less like models and more like software. They run multi-step loops, call tools, keep conversation state, can run for a long time, and change often as prompts and tools evolve. Packaging all of that as a static model artifact adds friction, while treating it as an application fits how agents are actually built and changed. For this reason, Databricks recommends Apps for new custom agents and publishes a migration path from Model Serving.

**What stays the same either way:**
- MLflow tracing and evaluation work the same on both.
- Agents reach governed Databricks resources, such as Unity Catalog functions, AI Search, and foundation models, under scoped permissions.
- Model Serving remains the home for the LLMs and classic ML models that an agent calls. Apps hosts the agent; serving endpoints host the models behind it.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Takeaway</strong>
<p style="margin: 8px 0 0 0; color: #333;">Model Serving deploys an agent the way you would deploy a model. Databricks Apps deploys it the way you would deploy software. Choose based on whether your agent behaves more like a function you call or a service you run. For most new agents, it is the latter.</p>
</div>
</div>
</div>

Deploy an agent with Model Serving (<a href="https://docs.databricks.com/aws/en/agents/custom-agents/model-serving/deploy-agent" target="_blank" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/agents/custom-agents/model-serving/deploy-agent/" target="_blank" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/agents/custom-agents/model-serving/deploy-agent" target="_blank" style="color: #1976d2; text-decoration: underline;">GCP</a>)

Migrate an agent from Model Serving to Databricks Apps (<a href="https://docs.databricks.com/aws/en/agents/custom-agents/migrate-agent-to-apps" target="_blank" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/agents/custom-agents/migrate-agent-to-apps/" target="_blank" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/agents/custom-agents/migrate-agent-to-apps" target="_blank" style="color: #1976d2; text-decoration: underline;">GCP</a>)

Deploy agents on Databricks (<a href="https://docs.databricks.com/aws/en/agents/deploy" target="_blank" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/agents/deploy/index/" target="_blank" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/agents/deploy" target="_blank" style="color: #1976d2; text-decoration: underline;">GCP</a>)

<div style="width: 100%; font-family: sans-serif;">

<div style="margin: 0 4px; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 16px 18px 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07);">

  <div style="height: 5px; border-radius: 5px; margin: -4px 0 12px; background: #FF5F46;"></div>

  <img src="https://files.training.databricks.com/binder/prod_main/deploying-and-monitoring-agents-on-databricks-en_us-1.1.0/images/20261029T133314Z/Deploying and Monitoring Agents on Databricks/Includes/images/genie-code.png" style="height: 64px; margin-bottom: 10px;">

  <div style="font-size: 15pt; color: #0b2026; line-height: 1.7; margin-bottom: 16px;">
    Want to learn more about <strong>agent deployment patterns</strong>? Ask Genie Code. Click on the genie icon <img src="https://files.training.databricks.com/binder/prod_main/deploying-and-monitoring-agents-on-databricks-en_us-1.1.0/images/20261029T133314Z/Deploying and Monitoring Agents on Databricks/Includes/images/genie-icon.png" style="height: 32px; vertical-align: middle;"> and begin querying. For example, click the <strong>Copy</strong> button below and paste into <strong>Genie Code</strong>.
  </div>

  <div style="display: flex; align-items: center; gap: 10px; background: #F9F7F4; border: 1px solid #ddd; border-radius: 6px; padding: 10px 14px; font-size: 14pt; font-family: monospace; color: #0b2026;">
    <span id="genie-query" style="flex: 1;">What are the differences between deploying an agent as a Databricks App versus a model serving endpoint?</span>
    <button onclick="
      var text = document.getElementById('genie-query').innerText;
      var ta = document.createElement('textarea');
      ta.value = text;
      ta.style.position = 'fixed';
      ta.style.opacity = '0';
      document.body.appendChild(ta);
      ta.select();
      document.execCommand('copy');
      document.body.removeChild(ta);
      this.innerText = 'Copied!';
      var btn = this;
      setTimeout(function(){ btn.innerText = 'Copy'; }, 2000);
    " style="background: #FF5F46; color: white; border: none; border-radius: 4px; padding: 4px 12px; font-size: 13pt; cursor: pointer; white-space: nowrap;">Copy</button>
  </div>

</div>

</div>

### B4. Resource Injection

The `databricks.yml` bundle file declares every resource the app depends on. The `app.yaml` file controls how those resources are injected into the running app as **environment variables** via its top-level `env:` section. This means agent code reads configuration from the environment rather than hardcoding workspace-specific values.

Here is how the three files work together:

<div style="background: #fff; max-width: 1600px; margin: 0 auto; padding: 12px 0 24px;">
<svg viewBox="0 0 1600 510" width="100%" role="img" aria-label="Resource injection in three files. databricks.yml declares a resource with the key model-service that grants the app EXECUTE on workspace.agent_deployment.agent-model. app.yaml sets the MODEL_SERVICE environment variable from that key with valueFrom. agent.py reads MODEL_SERVICE and passes it to the agent as its model. At runtime, MODEL_SERVICE holds the model service&#x27;s full name." style="display: block; height: auto; font-family: -apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Arial,sans-serif;">
<defs>
<marker id="ri-ar" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="8" markerHeight="8" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#618794"/></marker>
<filter id="ri-sh" x="-10%" y="-10%" width="120%" height="130%"><feDropShadow dx="0" dy="3" stdDeviation="6" flood-color="#1B3139" flood-opacity="0.08"/></filter>
</defs>
<rect x="40" y="20" width="560" height="332" rx="18" fill="#303F47" filter="url(#ri-sh)"/>
<rect x="62" y="36" width="516" height="5" rx="2.5" fill="#FF5F46"/>
<text x="62" y="72" font-size="21" font-weight="700" fill="#fff" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">databricks.yml</text>
<text x="62" y="116" font-size="18" font-weight="400" fill="#E6EBEE" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">resources:</text>
<text x="62" y="147" font-size="18" font-weight="400" fill="#E6EBEE" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">  - name: <tspan style="font-family: inherit;" fill="#FF8774">model-service</tspan></text>
<text x="62" y="178" font-size="18" font-weight="400" fill="#E6EBEE" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">    uc_securable:</text>
<text x="62" y="209" font-size="18" font-weight="400" fill="#E6EBEE" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">      securable_full_name:</text>
<text x="62" y="240" font-size="18" font-weight="400" fill="#E6EBEE" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">        <tspan style="font-family: inherit;" fill="#FFCC66">workspace.agent_deployment</tspan></text>
<text x="62" y="271" font-size="18" font-weight="400" fill="#E6EBEE" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">        <tspan style="font-family: inherit;" fill="#FFCC66">.agent-model</tspan></text>
<text x="62" y="302" font-size="18" font-weight="400" fill="#E6EBEE" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">      securable_type: MODEL_SERVICE</text>
<text x="62" y="333" font-size="18" font-weight="400" fill="#E6EBEE" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">      permission: EXECUTE</text>
<rect x="660" y="20" width="420" height="332" rx="18" fill="#303F47" filter="url(#ri-sh)"/>
<rect x="682" y="36" width="376" height="5" rx="2.5" fill="#2272B4"/>
<text x="682" y="72" font-size="21" font-weight="700" fill="#fff" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">app.yaml</text>
<text x="682" y="116" font-size="18" font-weight="400" fill="#E6EBEE" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">env:</text>
<text x="682" y="147" font-size="18" font-weight="400" fill="#E6EBEE" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">  - name: <tspan style="font-family: inherit;" fill="#8FD6B4">MODEL_SERVICE</tspan></text>
<text x="682" y="178" font-size="18" font-weight="400" fill="#E6EBEE" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">    valueFrom: <tspan style="font-family: inherit;" fill="#FF8774">model-service</tspan></text>
<rect x="1140" y="20" width="420" height="332" rx="18" fill="#303F47" filter="url(#ri-sh)"/>
<rect x="1162" y="36" width="376" height="5" rx="2.5" fill="#00A972"/>
<text x="1162" y="72" font-size="21" font-weight="700" fill="#fff" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">agent.py</text>
<text x="1162" y="116" font-size="18" font-weight="400" fill="#E6EBEE" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">MODEL_SERVICE = os.getenv(</text>
<text x="1162" y="147" font-size="18" font-weight="400" fill="#E6EBEE" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">    <tspan style="font-family: inherit;" fill="#FFCC66">&quot;</tspan><tspan style="font-family: inherit;" fill="#8FD6B4">MODEL_SERVICE</tspan><tspan style="font-family: inherit;" fill="#FFCC66">&quot;</tspan>)</text>
<text x="1162" y="178" font-size="18" font-weight="400" fill="#E6EBEE" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve"></text>
<text x="1162" y="209" font-size="18" font-weight="400" fill="#E6EBEE" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">agent = Agent(</text>
<text x="1162" y="240" font-size="18" font-weight="400" fill="#E6EBEE" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">    model=MODEL_SERVICE,</text>
<text x="1162" y="271" font-size="18" font-weight="400" fill="#E6EBEE" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">    <tspan style="font-family: inherit;" fill="#9FB2BB">...</tspan></text>
<text x="1162" y="302" font-size="18" font-weight="400" fill="#E6EBEE" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">)</text>
<path d="M606,186.0 H652" fill="none" stroke="#618794" stroke-width="2.4" marker-end="url(#ri-ar)"/>
<path d="M1086,186.0 H1132" fill="none" stroke="#618794" stroke-width="2.4" marker-end="url(#ri-ar)"/>
<text x="320" y="390" font-size="17" font-weight="600" fill="#5A6F77" text-anchor="middle" xml:space="preserve">Grants access</text>
<text x="870" y="390" font-size="17" font-weight="600" fill="#5A6F77" text-anchor="middle" xml:space="preserve">Injects the name</text>
<text x="1350" y="390" font-size="17" font-weight="600" fill="#5A6F77" text-anchor="middle" xml:space="preserve">Reads it</text>
<rect x="40" y="422" width="1520" height="64" rx="14" fill="#F9F7F4" stroke="#EEEDE9"/>
<text x="68" y="461" font-size="15" font-weight="700" fill="#618794" text-anchor="start" letter-spacing="1.4" xml:space="preserve">AT RUNTIME</text>
<text x="220" y="462" font-size="20" font-weight="600" fill="#0b2026" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">MODEL_SERVICE = <tspan style="font-family: inherit;" fill="#2272B4">&quot;workspace.agent_deployment.agent-model&quot;</tspan></text>
</svg>
</div>

<details style="margin: 14px 4px; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 0 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07); font-family: sans-serif;">
<summary style="cursor: pointer; padding: 16px 0; font-weight: 700; font-size: 18px; color: #0b2026;">Example: databricks.yml resource declaration</summary>
<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
<div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
  <div style="display:inline-flex;align-items:center;gap:0.45rem;">
    <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">YAML</span>
  </div>
  <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
    <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
    <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
    <span class="copy-label">Copy</span>
  </button>
</div>
<div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;"># databricks.yml -- declares the app and its resources
bundle:
  name: my-agent-app
<span></span>
resources:
  apps:
    my_agent:
      name: my-agent-app
      source_code_path: ./src
      resources:
        - name: model-service
          uc_securable:
            securable_full_name: workspace.agent_deployment.agent-model
            securable_type: MODEL_SERVICE
            permission: EXECUTE
      config:
        command: ["uv", "run", "start-server"]
        env:
          - name: DATABRICKS_HOST
            value: ${workspace.host}
          - name: MLFLOW_EXPERIMENT_NAME
            value: /Users/me/my-experiment</div>
</div>
</details>

<details style="margin: 14px 4px; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 0 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07); font-family: sans-serif;">
<summary style="cursor: pointer; padding: 16px 0; font-weight: 700; font-size: 18px; color: #0b2026;">Example: app.yaml environment variable injection</summary>
<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
<div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
  <div style="display:inline-flex;align-items:center;gap:0.45rem;">
    <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">YAML</span>
  </div>
  <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
    <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
    <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
    <span class="copy-label">Copy</span>
  </button>
</div>
<div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;"># app.yaml -- runtime manifest with env var injection
# valueFrom references a resource key defined in the app's resources block
command: ["uv", "run", "start-server"]
env:
  - name: MODEL_SERVICE
    valueFrom: "model-service"
  - name: SECRET_KEY
    valueFrom: "my-secret"
  - name: MLFLOW_EXPERIMENT_NAME
    value: "/Users/me/my-experiment"</div>
</div>
</details>

<details style="margin: 14px 4px; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 0 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07); font-family: sans-serif;">
<summary style="cursor: pointer; padding: 16px 0; font-weight: 700; font-size: 18px; color: #0b2026;">Example: Reading environment variables in agent code</summary>
<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
<div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
  <div style="display:inline-flex;align-items:center;gap:0.45rem;">
    <svg width="18" height="18" viewBox="0 0 24 24" fill="none"><path d="M12 2C8 2 8 4.5 8 4.5V7h4v1H6.5S3 8 3 12.5 5.5 17 5.5 17H8v-3.5S7.8 11 10 11h4s2 .1 2-2V4.5S16 2 12 2zm-1 1.8a.9.9 0 110 1.8.9.9 0 010-1.8z" fill="#5A6F77"/><path d="M12 22c4 0 4-2.5 4-2.5V17h-4v-1h5.5s3.5 0 3.5-4.5S18.5 7 18.5 7H16v3.5s.2 2.5-2 2.5h-4s-2-.1-2 2v4.5S8 22 12 22zm1-1.8a.9.9 0 110-1.8.9.9 0 010 1.8z" fill="none" stroke="#5A6F77" stroke-width="1.5"/></svg>
    <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">Python</span>
  </div>
  <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
    <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
    <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
    <span class="copy-label">Copy</span>
  </button>
</div>
<div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;"># Reading injected resources in agent code
import os
<span></span>
model_service = os.environ["MODEL_SERVICE"]
experiment_name = os.environ["MLFLOW_EXPERIMENT_NAME"]
workspace_host = os.environ["DATABRICKS_HOST"]</div>
</div>
</details>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">Resource injection decouples agent logic from infrastructure configuration. The same agent code can run in development, staging, and production environments by changing only the <code>databricks.yml</code> resource declarations. In <code>databricks.yml</code>, environment variables are nested under <code>config.env</code>. In <code>app.yaml</code>, they use a top-level <code>env:</code> section. The <code>valueFrom</code> field in <code>app.yaml</code> references resource keys defined in the app's <code>resources</code> block. Note that <code>DATABRICKS_HOST</code> is automatically available in all Databricks apps, so it only needs explicit injection when using bundle substitution in <code>databricks.yml</code>.</p>
</div>
</div>
</div>

App environment variables and resource injection (<a href="https://docs.databricks.com/aws/en/dev-tools/databricks-apps/environment-variables" target="_blank" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/dev-tools/databricks-apps/environment-variables" target="_blank" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/dev-tools/databricks-apps/environment-variables" target="_blank" style="color: #1976d2; text-decoration: underline;">GCP</a>)

### B5. Unity Gateway

When a deployed agent calls a **model service** for inference, the request goes through **Unity Gateway**, the centralized AI governance layer on Databricks. A model service is a Unity Catalog object that routes requests to a foundation model. The gateway sits between your application and the models, providing guardrails, rate limiting, usage tracking, and traffic management without requiring changes to your agent code.

<div style="background: #fff; max-width: 1600px; margin: 0 auto; padding: 12px 0 24px;">
<svg viewBox="0 0 1600 398" width="100%" role="img" aria-label="Unity Gateway between callers and models. An agent app calls the agent-model service and the scorers call the judge-model service, both in workspace.agent_deployment. The gateway applies rate limits, guardrails, usage tracking, and fallbacks, then routes agent-model to databricks-gpt-6-1-sol and judge-model to databricks-gpt-6-luna in system.ai." style="display: block; height: auto; font-family: -apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Arial,sans-serif;">
<defs>
<marker id="gw-ar" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="8" markerHeight="8" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#618794"/></marker>
<filter id="gw-sh" x="-10%" y="-10%" width="120%" height="130%"><feDropShadow dx="0" dy="3" stdDeviation="6" flood-color="#1B3139" flood-opacity="0.08"/></filter>
</defs>
<rect x="470" y="20" width="640" height="358" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#gw-sh)"/>
<rect x="492" y="34" width="596" height="5" rx="2.5" fill="#00A972"/>
<rect x="492" y="53" width="142" height="26" rx="13.0" fill="#00A972"/><text x="563" y="71" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">UNITY GATEWAY</text>
<rect x="40" y="94" width="340" height="96" rx="12" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#gw-sh)"/>
<rect x="62" y="110" width="97" height="26" rx="13.0" fill="#FF5F46"/><text x="110" y="128" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">YOUR APP</text>
<text x="62" y="168" font-size="19" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">my-agent-app</text>
<rect x="492" y="94" width="596" height="96" rx="10" fill="#F9F7F4" stroke="#EEEDE9"/>
<text x="516" y="132" font-size="17" font-weight="500" fill="#5A6F77" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">workspace.agent_deployment</text>
<text x="516" y="168" font-size="22" font-weight="700" fill="#0b2026" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">agent-model</text>
<rect x="914" y="129" width="142" height="26" rx="13.0" fill="#2272B4"/><text x="985" y="147" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">MODEL SERVICE</text>
<rect x="1200" y="94" width="360" height="96" rx="12" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#gw-sh)"/>
<text x="1222" y="132" font-size="17" font-weight="500" fill="#5A6F77" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">system.ai</text>
<text x="1222" y="168" font-size="19" font-weight="700" fill="#0b2026" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">databricks-gpt-6-1-sol</text>
<path d="M386,142.0 H484" fill="none" stroke="#618794" stroke-width="2.4" marker-end="url(#gw-ar)"/>
<path d="M1094,142.0 H1192" fill="none" stroke="#618794" stroke-width="2.4" marker-end="url(#gw-ar)"/>
<rect x="40" y="206" width="340" height="96" rx="12" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#gw-sh)"/>
<rect x="62" y="222" width="79" height="26" rx="13.0" fill="#1B5162"/><text x="101" y="240" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">JUDGES</text>
<text x="62" y="280" font-size="19" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">Scorers</text>
<rect x="492" y="206" width="596" height="96" rx="10" fill="#F9F7F4" stroke="#EEEDE9"/>
<text x="516" y="244" font-size="17" font-weight="500" fill="#5A6F77" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">workspace.agent_deployment</text>
<text x="516" y="280" font-size="22" font-weight="700" fill="#0b2026" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">judge-model</text>
<rect x="914" y="241" width="142" height="26" rx="13.0" fill="#2272B4"/><text x="985" y="259" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">MODEL SERVICE</text>
<rect x="1200" y="206" width="360" height="96" rx="12" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#gw-sh)"/>
<text x="1222" y="244" font-size="17" font-weight="500" fill="#5A6F77" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">system.ai</text>
<text x="1222" y="280" font-size="19" font-weight="700" fill="#0b2026" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">databricks-gpt-6-luna</text>
<path d="M386,254.0 H484" fill="none" stroke="#618794" stroke-width="2.4" marker-end="url(#gw-ar)"/>
<path d="M1094,254.0 H1192" fill="none" stroke="#618794" stroke-width="2.4" marker-end="url(#gw-ar)"/>
<rect x="492" y="322" width="133" height="34" rx="17.0" fill="#E7F6EF" stroke="#00A972" stroke-width="1.2"/><text x="558" y="345" font-size="17" font-weight="700" fill="#00A972" text-anchor="middle" xml:space="preserve">Rate limits</text>
<rect x="637" y="322" width="123" height="34" rx="17.0" fill="#E7F6EF" stroke="#00A972" stroke-width="1.2"/><text x="698" y="345" font-size="17" font-weight="700" fill="#00A972" text-anchor="middle" xml:space="preserve">Guardrails</text>
<rect x="772" y="322" width="161" height="34" rx="17.0" fill="#E7F6EF" stroke="#00A972" stroke-width="1.2"/><text x="853" y="345" font-size="17" font-weight="700" fill="#00A972" text-anchor="middle" xml:space="preserve">Usage tracking</text>
<rect x="945" y="322" width="114" height="34" rx="17.0" fill="#E7F6EF" stroke="#00A972" stroke-width="1.2"/><text x="1002" y="345" font-size="17" font-weight="700" fill="#00A972" text-anchor="middle" xml:space="preserve">Fallbacks</text>
</svg>
</div>

Key capabilities of Unity Gateway:

| Capability | Description |
|---|---|
| **Guardrails** | Enforce safety and content policies on requests and responses at the gateway level |
| **Rate limiting** | Configure consumption limits per model service to manage capacity and control costs |
| **Usage tracking** | Analyze costs by model service, model, principal, and tags via billable usage system tables |
| **Traffic splitting** | Distribute requests across multiple model backends for A/B testing or gradual rollouts |
| **Inference tables** | Log requests and responses to Unity Catalog Delta tables for auditing and analysis |

The gateway governs more than just agent traffic. It provides a unified governance layer across LLM endpoints, MCP servers, and coding agents (such as Cursor, Claude Code, and Codex CLI), giving platform teams a single place to manage AI access across the organization.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">Your agent code only names the model service, for example with <code>DatabricksOpenAI(use_ai_gateway=True)</code> and <code>model="catalog.schema.agent-model"</code>. Rate limits, guardrails, and usage tracking are configured on the service, so changing them needs no code changes.</p>
</div>
</div>
</div>

Unity Gateway overview (<a href="https://docs.databricks.com/aws/en/ai-gateway/" target="_blank" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/ai-gateway/" target="_blank" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/ai-gateway/" target="_blank" style="color: #1976d2; text-decoration: underline;">GCP</a>)

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
